<img src="https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/images/edrai_logo.png" alt="EDR|AI" width="300"/>

# Chapter 46 — Evidence Synthesis: Scoping Reviews and Small Meta-Analyses

This is the **companion notebook** of [Chapter 46 — Evidence Synthesis: Scoping Reviews and Small Meta-Analyses](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/evidence-synthesis.html) from **EDR|AI — Evidence-Driven Research in the Age of AI**. Authored by [Davi Moreira](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html).

[Open the chapter](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/part7-further-routes/evidence-synthesis.html) · [Book home](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/index.html) · [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html)

*AI is your arm and your research assistant, not your brain.*

## How to use this notebook

1. Work top to bottom, with the chapter open in another tab.
2. Copy each **AI prompt** into your AI tool, run it, then record in the response cell what came back and what you verified.
3. Run the code cells; change something and run again.
4. Finish the **It is your turn** workspace at the end — that is this chapter's step of your own research project.
5. Log every AI use in your **AI Research Ledger**: task · tool · prompt · output summary · decision · verification method · remaining concern · you as the responsible researcher.
6. Your AI can be more than a chatbot: agentic tools can run multi-step work for you. Delegating boldly is fine; reviewing, curating, and deciding stay yours.

> **The research decision.** Decide which studies count before you see a single
> result, how you will find every one of them, and whether combining them answers
> your question or only averages their biases. A tool can search, sort, and pool
> in seconds, but only you can defend why each study is in the pile and what the
> pooled number is allowed to mean.

## Code from the chapter

The cells below come from the chapter. Run them, then change something and run again — the numbers should move the way the chapter says they will.

*From the section “A worked example”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats

# Six canvassing experiments, CONSTRUCTED for this example. Each row is one
# study's effect on support for the candidate (percentage points) and its
# standard error, copied from the study's own results table. One row per study:
# where a study reported several estimates, the protocol's rule picked one.
studies = pd.DataFrame({
    "study":    ["A", "B", "C", "D", "E", "F"],
    "estimate": [2.9, 0.6, -0.4, 2.4, 0.3, 1.1],
    "se":       [1.4, 0.8, 1.1, 1.7, 0.45, 0.9],
})

def pool(y, s):
    k = len(y)
    w = 1 / s**2                                   # precise studies weigh more
    fixed = np.sum(w * y) / np.sum(w)
    fixed_ci = fixed + np.array([-1.96, 1.96]) / np.sqrt(np.sum(w))
    q = np.sum(w * (y - fixed) ** 2)               # disagreement beyond chance?
    tau2 = max(0.0, (q - (k - 1)) / (np.sum(w) - np.sum(w**2) / np.sum(w)))
    wr = 1 / (s**2 + tau2)                         # weights that allow for the spread
    rand = np.sum(wr * y) / np.sum(wr)
    se_z = 1 / np.sqrt(np.sum(wr))
    z_ci = rand + np.array([-1.96, 1.96]) * se_z   # normal approximation: too narrow
    # Hartung-Knapp-Sidik-Jonkman: rescale by the scatter the studies show (never
    # below the usual standard error) and use a t multiplier on k - 1 df.
    scatter = np.sum(wr * (y - rand) ** 2) / (k - 1)
    se_h = np.sqrt(max(scatter, 1.0)) * se_z
    hk_ci = rand + np.array([-1, 1]) * stats.t.ppf(0.975, k - 1) * se_h
    # Prediction interval: where the true effect in ONE new setting may fall.
    pred = rand + np.array([-1, 1]) * stats.t.ppf(0.975, k - 2) * np.sqrt(tau2 + se_z**2)
    i2 = max(0.0, (q - (k - 1)) / q) * 100
    return dict(fixed=fixed, fixed_ci=fixed_ci, rand=rand, hk_ci=hk_ci,
                z_ci=z_ci, pred=pred, i2=i2)

def show(ci):
    return f"[{ci[0]:+.2f}, {ci[1]:+.2f}]"

y, s = studies.estimate.to_numpy(), studies.se.to_numpy()
sig = (np.abs(y / s) > 1.96).sum()
r = pool(y, s)
print(f"vote count (significant studies)  : {sig} of {len(y)}")
print(f"fixed-effect pooled estimate      : {r['fixed']:+.2f}  {show(r['fixed_ci'])}")
print(f"random-effects pooled estimate    : {r['rand']:+.2f}  {show(r['hk_ci'])}")
print(f"  same, normal-approximation CI   :        {show(r['z_ci'])}")
print(f"prediction interval, new setting  :        {show(r['pred'])}")
print(f"share of spread beyond chance (I2): {r['i2']:.0f}%")

# One extraction slip: study E's outcome STANDARD DEVIATION (48 points, from
# its descriptive table) typed into the standard-error column.
s_bad = s.copy(); s_bad[4] = 48
b = pool(y, s_bad)
print(f"\nsame pool, E's SD typed as its SE : {b['rand']:+.2f}  {show(b['hk_ci'])}")
print(f"  with the normal-approximation CI:        {show(b['z_ci'])}")
print("the largest study vanished from the average and nothing crashed")

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

*From the section “A seeded simulation”.* **What this cell does:** exactly what the chapter walks through in that section; run it and compare with the chapter.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEED = 464
rng = np.random.default_rng(SEED)

k, mu, tau = 30, 1.0, 0.5            # 30 studies RUN; true average +1.0 point
se = rng.uniform(0.3, 2.5, k)        # small studies carry large standard errors
theta = rng.normal(mu, tau, k)       # each study's own true effect
est = rng.normal(theta, se)          # what each study reports
significant = est / se > 1.96
published = significant | (rng.random(k) < 0.25)   # nulls rarely get out

def pool(y, s):                      # random-effects average, HKSJ interval
    k = len(y)
    w = 1 / s**2
    fe = np.sum(w * y) / np.sum(w)
    q = np.sum(w * (y - fe) ** 2)
    t2 = max(0.0, (q - (k - 1)) / (np.sum(w) - np.sum(w**2) / np.sum(w)))
    wr = 1 / (s**2 + t2)
    m = np.sum(wr * y) / np.sum(wr)
    scatter = np.sum(wr * (y - m) ** 2) / (k - 1)
    half = stats.t.ppf(0.975, k - 1) * np.sqrt(max(scatter, 1.0) / np.sum(wr))
    return m, m - half, m + half

all_m, all_lo, all_hi = pool(est, se)
pub_m, lo, hi = pool(est[published], se[published])
print(f"published: {published.sum()} of {k}, significant: {significant.sum()}")
print(f"pooled, all run   : {all_m:.2f} [{all_lo:.2f}, {all_hi:.2f}]")
print(f"pooled, published : {pub_m:.2f} [{lo:.2f}, {hi:.2f}]")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.6, 4.2))
idx = np.where(published)[0]
idx = idx[np.argsort(se[idx])]                     # most precise on top
ys = np.arange(len(idx), 0, -1) + 1.0
a1.errorbar(est[idx], ys, xerr=1.96 * se[idx], fmt="s", color="#2a78d6", ms=4)
a1.fill([lo, pub_m, hi, pub_m], [0.5, 0.85, 0.5, 0.15], color="#eb6834")
a1.axvline(0, color="#9a9a9a", lw=0.8)                 # zero effect
a1.axvline(mu, color="#333333", ls="--")               # the truth
a1.set_xlabel("Effect on support (percentage points)")
a1.set_title("What the published record shows", loc="left")
a2.scatter(est[published], se[published], color="#2a78d6", label="published")
a2.scatter(est[~published], se[~published], facecolors="none",
           edgecolors="#9a9a9a", label="never published")
grid = np.linspace(0, se.max() * 1.05, 50)
a2.plot(1.96 * grid, grid, color="#9a9a9a", ls=":", label="significance line")
a2.axvline(pub_m, color="#eb6834", label="pooled, published")
a2.axvline(all_m, color="#2a78d6", label="pooled, all run")
a2.axvline(mu, color="#333333", ls="--"); a2.invert_yaxis()
a2.set_xlabel("Effect on support (percentage points)")
a2.set_ylabel("Standard error"); a2.legend()
a2.set_title("Every study that was run", loc="left")
plt.show()

**Reading the output.** The chapter reads this output in the same section; check yours against it, then change one input and rerun. The numbers should move the way the chapter says they will.

## It is your turn

<!-- station-pointer:begin -->
> **A further route beyond the five pathways.** This lesson
> extends [Studio 5: Develop the pathway](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/studio05-develop-the-pathway.html). Read it once
> you have declared your primary pathway and your question
> calls for this design. Studio 5's milestone asks for the
> same decisions, answered for this route.
<!-- station-pointer:end -->

*Your question has a literature, and you have chosen to make that literature your
data. This step writes the review's own Research Contract, so every study in your
pile has a documented reason to be there.*

The hands-on half of this section lives in the chapter's **companion notebook**: open it in Colab with the badge at the top, and work the steps there.

Commit your own answer first, then delegate. Each prompt is a checkable job, not a
request for a verdict. Work them as a loop. The first answer is a draft: find the
claim you cannot check, say so in your next message, and run it again. Some tools
will run that whole loop unattended and hand you a finished review. The finished
look is exactly what makes the questions in this chapter worth asking before you
accept it.

> **Do not delegate.**
>
> Three calls stay yours. You decide **which studies are eligible, in writing, before
> you read any result**, because a rule set after the results is a filter on the
> answer. You decide **whether pooling is defensible at all**, because only you can
> judge whether the studies answer one question with comparable designs and outcomes.
> And you decide **what the synthesis can claim**, since it inherits every study's
> warrant and no more. A tool can search, screen a first pass, and compute a weighted
> average in seconds. You own the eligibility rules, the decision to pool, and the
> sentence that reports it.

**Step 1.** Write your review question and its eligibility rules before you search. Use
PICO for an effect question or PCC for a scoping question, and add the designs
you accept, the date range, the languages, the reasons you will use to exclude a
full text, and your rule for choosing one estimate when a study reports several. Then write your Research Contract
lines for this route: the question; the estimand (the average true effect and its
spread, or for a scoping review the map you will produce); the data strategy
(search, screening, extraction); the answer strategy (pooling or a structured
narrative); the warrant (the studies' own crossing licenses, never the pooling);
and the limits (which settings and which unpublished work you may be missing).

*When you are ready to delegate this step:*

```text
Act as a systematic-review methodologist. Here is my review question and my
eligibility rules: [paste them]. List, as a table, every term in my rules that two
careful screeners could read differently, and for each one give an example study
that one would include and the other would exclude. Do not rewrite my rules for me.
```

*After running, verify: read each flagged ambiguity against your own question and
tighten only the rules where the example would really split two screeners. If the
tool finds no ambiguity at all, ask again with "assume two screeners disagree on
10 percent of records; which rule causes it?" Counters **illusion of
completeness**, a tidy table that implies your rules have no gaps.*

✍️ **Your work for step 1.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 2.** Build a search log you could hand to a stranger. Name each database or source,
the exact search string, the date, and the number of records it returned, and
add at least one route to unpublished work.

*When you are ready to delegate this step:*

```text
Act as a research librarian in political science. My review question is
[paste it]. Suggest the databases and registries where studies on it are indexed,
and draft one search string for each using its own syntax. Only name databases
you are confident exist; mark anything uncertain.
```

*After running, verify: run every string yourself, record the hit count and date
in your log, and drop any database you cannot open. Counters **stale knowledge**
(a database renamed, merged, or retired since the tool learned about it).*

✍️ **Your work for step 2.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 3.** Screen in two stages with the rules you fixed in step 1. At titles and abstracts,
send anything unclear forward. At full texts, give every excluded study one
reason from your list and log it. Then make a documented second check: ideally a
second person screens every record, and at minimum a random 20 percent at each
stage, blind to your calls, and you log the agreement and how each disagreement was settled. If no
second person is available, say in your report that one person screened. Before
a tool proposes calls on anything, test it on a random sample you have already
screened blind, and never let it exclude a record you have not read.

*When you are ready to delegate this step:*

```text
Here are my eligibility rules [paste them] and 50 titles and abstracts [paste
them]. For each record, say include, exclude, or unsure, and quote the exact words
in the abstract that decided it. Do not decide a record from its title alone.
```

*After running, verify: compare the tool's calls with the ones you made blind, and
read every disagreement in full. Record the agreement rate in your log, and count
the eligible records the tool would have excluded: if there are any, use it only
to order your reading, never to exclude. Counters
**correlated errors**, a checker who reads the tool's answer first and inherits its
mistakes.*

✍️ **Your work for step 3.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 4.** Extract every study onto one form and trace each number to a page. For each
study, record the design, the setting, the estimate, its standard error or
interval, and your risk-of-bias note, with the page and table where each number
sits. Keep one row per study: when a study reports several estimates for your
question, apply the rule from step 1 and list the others in a note, never as new
rows.

*When you are ready to delegate this step:*

```text
Act as a careful research assistant. From the attached study, fill this extraction
form: [paste the form]. For every number, give the page and table it came from and
say whether it is a standard error, a standard deviation, or a confidence interval.
If a field is not reported, write "not reported"; never estimate it.
```

*After running, verify: open the page cited for every number and confirm it
exists, says that, and is the quantity the column asks for. Counters **confident
fabrication**, a number or a study that appears in no source you can open.*

✍️ **Your work for step 4.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 5.** Decide whether to pool, and write the synthesis sentence. If you pool, report the
pooled estimate with an interval built for few studies (the
Hartung-Knapp-Sidik-Jonkman interval), a prediction interval if you have at least
five studies, the heterogeneity, and your funnel-plot note;
if you do not, write the map and one sentence on why one number would mislead.
Either way, name the warrant the studies carry and the settings the answer
reaches. If you found only a handful of eligible studies, or they disagree on
design and outcome, the map is your product.

*When you are ready to delegate this step:*

```text
Here is my synthesis sentence: "[paste it]" and the studies behind it: [paste the
extraction table]. Act as a hostile review-journal editor. List every word that
claims more than these designs, this interval, and these settings allow. Do not
rewrite the sentence for me.
```

*After running, verify: check each flagged word against the studies' own designs
and your interval, and keep only the objections the evidence forces. If every
objection is mild, push back with "assume the sentence is wrong; name the single
worst flaw." Counters **sycophantic agreement** (praise that reviews your ego, not
your evidence).*

✍️ **Your work for step 5.** Double-click this cell and write your answer here.

✍️ **Your run.** Double-click this cell and record: what the AI returned (one or two lines), what you verified and how, and your ledger row.

**Step 6.** Log the step in your AI Research Ledger, and verify at least one output with a
named method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html).
**Primary-source reading** is the natural check for this route: open the PDF
behind every extracted number. **Direct calculation** is the strong second check:
recompute the pooled estimate by hand from your verified table and land on the same
number. An AI reviewer may run the checks with you; the decision to accept or
reject stays yours.

✍️ **Your work for step 6.** Double-click this cell and write your answer here.

### The standard this section is held to

Use this as a self-check while you work. It is also the bar the same work meets later, once your project carries it. Each row: **0** missing, **1** attempted but incomplete, generic, or unverified, **2** complete, specific to your own project, and verified where a check applies. **14 points in all.**

| # | Criterion | 0–2 |
|---|---|---|
| Step 1 | Write your review question and its eligibility rules before you search | |
| Step 2 | Build a search log you could hand to a stranger | |
| Step 3 | Screen in two stages with the rules you fixed in step 1 | |
| Step 4 | Extract every study onto one form and trace each number to a page | |
| Step 5 | Decide whether to pool, and write the synthesis sentence | |
| Step 6 | Log the step in your AI Research Ledger, and verify at least one output with a named method from the Verification Guide | |
| + | Craft and verification record: AI use logged in your AI Research Ledger, claims stated with their uncertainty, and each key claim verified with a named method | |

In [ ]:
# Scratch space — use this cell for any code your steps need.

**Before you leave this notebook:** add today's rows to your AI Research Ledger, and verify your key claim with a named method from the [Verification Guide](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/verification-guide.html). AI can review AI — but the last decision is human.

This was the last of the further research routes. Carry what you wrote back to [Studio 5: Develop the pathway](https://davi-moreira.github.io/2026F_evidence_driven_research_purdue_HONR464/book/studios/studio05-develop-the-pathway.html): its milestone asks for the same decisions, answered for your route.